# Experiment 4: Latent Dirichlet Allocation (LDA) for Topic Modeling
**Course Outcome:** CO4 | **Cognitive Level:** Bloom's Taxonomy L3 (Apply)

### Objective:
To apply **Latent Dirichlet Allocation (LDA)** on a document collection and interpret discovered topics using **probabilistic (Variational Bayes)** and **sampling-based (Collapsed Gibbs Sampling)** methods.

### Expected Outcomes & Deliverables:
1. Preprocess raw text data (tokenization, lemmatization, stopword removal, bigrams).
2. Build document-term representations (Document-Term Matrix, Bag-of-Words, vocabulary).
3. Train LDA models using Variational Inference and Collapsed Gibbs Sampling.
4. Interpret discovered topic-word probability distributions and document classifications.
5. Produce visualizations (topic bar charts, document-topic heatmap, t-SNE projection).

In [ ]:
# Step 1: Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessor import TextPreprocessor
from src.vectorizer import DocumentTermRepresentation
from src.lda_model import LDATrainer
from src.gibbs_sampler import CollapsedGibbsLDA
from src.visualizer import TopicVisualizer

print("Environment initialized successfully!")

## 2. Load Multi-Domain Document Collection
We load a collection of 25 multi-domain documents across **AI & Machine Learning**, **Space Exploration**, **Renewable Energy**, **Medical Healthcare**, and **Financial Economics**.

In [ ]:
df_raw = pd.read_csv('data/raw_documents.csv')
print(f"Total Documents: {len(df_raw)}")
print(f"Domains: {df_raw['domain'].unique().tolist()}")
df_raw[['doc_id', 'domain', 'title']].head(10)

## 3. Text Preprocessing (Deliverable: Preprocessed Corpus)
Transforming unstructured text into cleaned, lemmatized tokens with bigram collocations.

In [ ]:
preprocessor = TextPreprocessor(min_token_len=3)
raw_texts = df_raw['text'].tolist()
tokenized_docs, joined_docs = preprocessor.preprocess_corpus(raw_texts, use_bigrams=True)

# Create deliverable dataframe
df_preprocessed = df_raw.copy()
df_preprocessed['tokens'] = tokenized_docs
df_preprocessed['token_count'] = [len(t) for t in tokenized_docs]

print(f"Preprocessing complete. Average tokens per document: {df_preprocessed['token_count'].mean():.1f}")
print("Sample preprocessed tokens from Document 1:")
print(tokenized_docs[0][:12])

## 4. Document-Term Representation (Outcome 2)
Constructing Document-Term Matrix (DTM) and analyzing vocabulary sparsity.

In [ ]:
dtm_builder = DocumentTermRepresentation(min_df=1, max_df=0.90)
dtm, vocab = dtm_builder.fit_transform_sklearn(joined_docs)

print(f"DTM Shape: {dtm.shape[0]} documents x {dtm.shape[1]} unique vocabulary terms")
print(f"Matrix Sparsity: {dtm_builder.get_sparsity():.2f}%")
dtm_builder.get_top_frequent_words(top_n=10)

## 5. Train LDA Model via Variational Inference (Outcome 3 & 4)
Fitting LDA model with $K=5$ topics using Variational Bayes optimization.

In [ ]:
n_topics = 5
lda_trainer = LDATrainer(n_topics=n_topics, learning_method='batch', max_iter=60, random_state=42)
lda_trainer.train_sklearn(dtm, vocab, doc_topic_prior=0.1, topic_word_prior=0.01)

metrics = lda_trainer.evaluate_model(dtm)
print(f"Log-Likelihood: {metrics['log_likelihood']:.2f}")
print(f"Model Perplexity: {metrics['perplexity']:.2f}")

## 6. Discovered Topics & Topic-Word Lists (Deliverable)

In [ ]:
topic_words = lda_trainer.get_top_topic_words(top_n=8)
topic_words_df = lda_trainer.get_topic_word_dataframe(top_n=8)

for topic_idx, words in topic_words.items():
    words_str = ", ".join([f"{w} ({p:.3f})" for w, p in words[:6]])
    print(f"Topic {topic_idx + 1}: {words_str}")

topic_words_df.head(15)

## 7. Document-Topic Classification (Dominant Topics)

In [ ]:
doc_dist_df = lda_trainer.get_document_topic_dataframe(df_raw['title'].tolist(), raw_texts)
doc_dist_df[['Doc_ID', 'Title', 'Dominant_Topic', 'Topic_Confidence']].head(12)

## 8. Sampling-Based Method: Collapsed Gibbs Sampling
Implementing MCMC topic discovery via Collapsed Gibbs Sampling.

In [ ]:
gibbs_lda = CollapsedGibbsLDA(n_topics=5, alpha=0.5, beta=0.1, n_iter=100, random_state=42)
gibbs_lda.fit(tokenized_docs, vocab=vocab)
gibbs_topics = gibbs_lda.get_top_topic_words(top_n=6)

print("Gibbs Sampling Discovered Topics:")
for k, words in gibbs_topics.items():
    print(f"Topic {k+1}: {", ".join([w[0] for w in words])}")

## 9. Visualizations (Deliverables)
- Topic-Word Distribution Bar Charts
- Document-Topic Heatmap
- 2D t-SNE Projection of Document Topic Clusters
- Gibbs Convergence Curve

In [ ]:
visualizer = TopicVisualizer(output_dir='results')

topic_labels = {
    0: "Artificial Intelligence",
    1: "Space Exploration",
    2: "Renewable Energy",
    3: "Medical Healthcare",
    4: "Financial Economics"
}

bar_path = visualizer.plot_topic_word_distributions(topic_words, topic_labels=topic_labels)
heatmap_path = visualizer.plot_document_topic_heatmap(lda_trainer.doc_topic_dist, df_raw['title'].tolist())
tsne_path = visualizer.plot_topic_clusters_tsne(lda_trainer.doc_topic_dist, df_raw['title'].tolist())
gibbs_path = visualizer.plot_gibbs_convergence(gibbs_lda.log_likelihood_history)

print(f"All visualization plots generated in results/ directory!")